# CLIP 和 对照视觉-语言预训练

OpenAI的CLIP提供了一个简单但足以驱动接下来5年发展的想法：将图像编码器和文本编码器对齐到相同的向量空间中内，使用来自网络的图像-标题对，以及对比损失进行训练。零监督样本。

## 问题描述

早期的CLIP版本是基于监督学习的，收集带标签的数据集，训练一个CNN来实现。但是标签是很贵的，而且带有主观偏好，不好在没有微调的情况下往新的任务前一。

但是网络上有成百上千万的图片标题数据对，免费！CLIP的答案是，将图片标题数据对当成一个匹配问题，输入N张图片、N个标题，学习如果让每个图片正确匹配到标题，在有N-1个负样本的情况下。不需要类型标签、人工标注，只是一个对比损失。

## 基本概念

### 双边的编码器

CLIP有两个塔：
- 图片编码器`f`： ViT 或者 ResNet，每张图片输出一个D维向量。
- 文本编码器`g`： 一个小的Transformer，每个标题输出一个D维向量。

两边的向量都归一化，然后计算余弦相似度。
```
S[i, j] = cos(f(x_i), g(y_j)) / tau
```
`tau`是一个可学习的温度系数

### InfoNCE 损失

CLIP 的损失是分别对行和列对称计算的交叉熵：
```
loss_i2t = CE(S, labels=identity)  # 每张图片相对于标题的交叉熵
loss_t2i = CE(S^T, labes=identity)  # 每个标题相对于图片的交叉熵
loss = 0.5 * (loss_i2t + loss_t2i)
```
CLIP训练每批次大小为32k，这个量级相当重要。

### 温度
`tau`控制softmax函数的尖锐程度。系数除以小值会被指数运算进一步放大，产生更锐利的分布。相反，更大的系统分布更柔和。CLIP 学习的是`log(1/tau)`，并且裁剪防止坍缩。SigLIP2 修复了`tau`的初值并且使用了一个可学习的偏置。

### 为什么SigLIP的sigmoid更好

Softmax针对整个行和列做归一化计算，通信量是指数级别的。

SigLIP直接在单元格处做sigmoid，损失是一个二分类问题————图片和标题成对吗？期望对角线上为正值，其他地方都为负值。损失函数
```
L = -1/N sum over(i, j) [
    y_ij log sigmoid(S[i, j]) + (1 - y_ij) log sigmoid(-S[i, j])
]
```
`y_ij = 1` if `i == j`

因此，SigLIP2可以进一步扩大到32k-512k的量级。

### 无样本分类

给N个分类名称，对每个类都构建文本模板`a photo of a {class}`

将每个模板用文本编码器编码，每张图片用图片编码器编码，让预测类型的相似度最大。所以实际上没有在目标分类上训练。

提示词模板也很重要，CLIP的原始文章使用了80多种提示词模板，然后取平均。

### 线性探针与微调

零样本是基线，线性探针（在冻结的CLIP特征之上为你的目标类别训练一个线性层）在域内任务胜过零样本。全量微调在域内任务胜过线形探针，但可能伤害零样本迁移。三种范式，三种取舍。

#### SigLIP2

- NaFlex。 处理任意长宽比和分辨率
- 更稠密的特征用于分割和深度估计，用作VLMs的冻结骨架。
- 支持多种语言
- 参数量比CLIp更大。

2026年默认是SigLIP，但是CLIP在纯图像-文本召回任务上仍有优势。
